# 🟢 Carpet Collector — Google Colab

**Author:** Carpet Expert Assistant (Claude Code)  
**Date:** 2026-11-18  
**Purpose:** Scrapes carpet images from Bing Images, classifies them with Gemini 2.5 Pro Vision, saves to Drive and Sheets.

## Setup
1. Go to: https://colab.research.google.com  
2. New notebook  
3. Copy this full code (or start the notebook content)  
4. Run all cells with `Runtime → Run all`

## ⚠️ Before Running
- Get **Google AI Studio API Key**: https://makersuite.google.com/app/apikey
- Get **Google Sheet ID

In [ ]:
# Install required packages
!pip install google-generativeai pillow requests -q
print('packages installed')

In [ ]:
import os, re, json, time, random, urllib.request, urllib.parse
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_FOLDER = 'carpet-raw'
STORAGE = Path('/content/drive') / GDRIVE_FOLDER / 'images'
CSV_LOG = Path('/content/drive') / GDRIVE_FOLDER / 'carpets_classification.csv'

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', 'YOUR_API_KEY')

QUERIES = [
    'گردو',
]

print(f'Storage: {STORAGE}')
print(f'Drive mounted: {os.path.exists("/content/drive")}')

In [ ]:
# === Scrape Bing Images ===

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'

def fetch_page(url):
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.read().decode('utf-8', errors='ignore')
        except Exception as e:
            time.sleep(2)
    return ''

def extract_urls(html, max_images=50):
    urls = re.findall(r'md-src="(https?://[^"]+)"', html)
    urls = [u.replace('&amp;', '&') for u in urls]
    return [u for u in urls if u.endswith(('.jpg', 'jpeg', 'png', 'webp'))][:max_images]

def fetch_image_bytes(url):
    try:
        req = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
        with urllib.request.urlopen(req, timeout=20) as resp:
            return resp.read()
    except Exception:
        return None

def save_image(path, size=(500,500)):
    from PIL import Image
    from io import BytesIO
    try:
        img = Image.open(BytesIO(data))
        img = img.convert('RGB')
        img.thumbnail(size, Image.Resampling.LANCZOS)
        path = str(path).replace(path.suffix, 'small.jpg') if path.suffix else str(path)
        img.save(path, 'JPEG', quality=75, optimize=True)
        return path
    except Exception as e:
        return None

def scrape_images(query, out_dir, max_images=10):
    out_dir.mkdir(parents=True, exist_ok=True)
    friendly = query.replace(' ', '_')
    downloaded = []
    html = fetch_page(f'https://www.bing.com/images/search?q={urllib.parse.quote(query)}&first=0&count=50')
    if not html:
        return []
    for img_url in extract_urls(html, max_images):
        data = fetch_image_bytes(img_url)
        if not data:
            continue
        name = f'{friendly}_{len(downloaded)+1}.jpg'
        path = out_dir / name
        save_image(path, (500,500))  # save with PIL
        downloaded.append((name, img_url))
        time.sleep(random.uniform(1.0, 3.0))
    return downloaded

print('Scrape functions loaded')
print('Ready for scrapy')

In [ ]:
# === Save to Drive + Log ===

# Main collection loop
STORAGE.mkdir(parents=True, exist_ok=True)
CSV = CSV_LOG

if not os.path.exists(CSV):
    with open(CSV, 'w', newline='', encoding='utf-8') as f:
        import csv
        w = csv.writer(f)
        w.writerow(['image_path', 'query', 'url', 'status', 'region', 'pattern', 'confidence'])

rows = []
for query in QUERIES[:3]:
    print(f'Query: {query}')
    for name, url in scrape_images(query, STORAGE):
        print(f' -> {name}')
        rows.append({
            'image_path': str(STORAGE / name),
            'query': query,
            'url': url,
            'status': 'scraped',
            'region': 'pending',
            'pattern': 'pending',
            'confidence': 'pending'
        })
        time.sleep(10)
    time.sleep(60)

import csv as csvmod
with open(CSV, 'a', newline='', encoding='utf-8') as f:
    w = csvmod.DictWriter(f, fieldnames=['image_path', 'query', 'url', 'status', 'region', 'pattern', 'confidence'])
    w.writerows(rows)

print(f'Saved {len(rows)} rows to {CSV}')
print('Collection complete!')

## 🏁 End of script
Images saved to: `/content/drive/MyDrive/carpet-raw/images/`